# Kwantyfikacja wspólnego rodowodu konstrukcyjnego we flocie transformatorów mocy za pomocą PROC INBREED

## Streszczenie

Transformatory rozdzielni sieciowej u operatora sieci są projektowane w kolejnych generacjach konstrukcyjnych, a każdy nowy model wywodzi się z dwóch poprzedzających go projektów. Ten notatnik traktuje tę genealogię inżynierską jako rodowód (pedigree) i wykorzystuje **PROC INBREED** do obliczenia współczynników inbredu oraz addytywnych współczynników pokrewieństwa (kowariancji), które kwantyfikują, jak wiele wspólnego rodowodu konstrukcyjnego dzielą dowolne dwa urządzenia.

Rodowód jest skonstruowany tak, aby struktura była łatwa do interpretacji: dwa z czterech obecnych modeli flotowych pochodzą od pary **rodzeństwa** wśród projektów-poprzedników i dlatego niosą skoncentrowany rodowód, podczas gdy pozostałe czerpią z rozłącznych linii. Procedura odtwarza to dokładnie. Dwa modele wywodzące się od rodzeństwa (`G3_T1`, `G3_T3`) mają współczynnik inbredu **F = 0,25**; pozostałe dwa (`G3_T2`, `G3_T4`) mają **F = 0**. Średni współczynnik inbredu dla całej floty wynosi **0,0417**. Odczytując macierz addytywnego pokrewieństwa, najmniej spokrewnioną parą w obecnej flocie jest **`G3_T1` i `G3_T3` (pokrewieństwo = 0)** — nie dzielą one żadnego wspólnego pochodzenia i tworzą najsilniejszą redundantną (N-1) parę, co ma znaczenie, ponieważ `G3_T1` sam w sobie jest jedną z najbardziej inbredowych konstrukcji.

## Źródła danych

| Zbiór danych | Wiersze | Kluczowe zmienne | Opis |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | Mały, deterministyczny inżynierski rodowód konstrukcji transformatorów rozdzielni w trzech nienakładających się generacjach projektowych (4 platformy założycielskie, 4 pochodne drugiej generacji, 4 obecne modele flotowe). Każda konstrukcja niebędąca założycielem zapisuje dwie odrębne konstrukcje-poprzedniki (`Parent1`, `Parent2`), z których się wywodzi. `Sex` oznacza wiodącą rolę konstrukcyjną (M = linia mechaniczna/rdzenia, F = linia elektryczna/uzwojenia). Rodowód jest zadany ręcznie — nie losowo — dzięki czemu struktura pokrewieństwa jest znana z góry i można ją zweryfikować względem wyniku procedury.

# Kwantyfikacja wspólnego rodowodu konstrukcyjnego we flocie transformatorów mocy

## Dlaczego zakład użyteczności publicznej dba o „inbred”

Operator przesyłu i dystrybucji energii eksploatuje setki transformatorów mocy w rozdzielniach. Aby kontrolować koszty inżynierskie i wysiłek kwalifikacyjny, producenci rzadko projektują każdy transformator od podstaw — każdy nowy model **dziedziczy** podstawową geometrię, topologię uzwojeń, systemy izolacji i konstrukcje przepustów od jednego lub dwóch poprzedzających modeli. Po kilku cyklach zamówień powstaje w ten sposób prawdziwa *genealogia inżynierska*: rodowód konstrukcji.

To wspólne dziedzictwo stanowi ukryte ryzyko niezawodnościowe. Jeśli dwa transformatory zabezpieczające to samo obciążenie (redundantna para **N-1**) wywodzą się z tej samej konstrukcji przodka, ukryta wada projektowa — mod rezonansowy, mechanizm starzenia izolacji, ścieżka przeskoku na przepuście — prawdopodobnie występuje w **obu** jednostkach. Pojedyncza przyczyna źródłowa może wtedy wyłączyć redundantną parę jednocześnie: *awaria wspólnego trybu*.

**PROC INBREED** została stworzona właśnie po to, aby kwantyfikować tego rodzaju wspólne pochodzenie. Choć jej korzenie tkwią w hodowli zwierząt i roślin, jej matematyka — rekurencja addytywnego pokrewieństwa Wrighta — jest niezależna od dziedziny: mierzy, jaką część dziedzictwa konstrukcyjnego dzielą dwa osobniki poprzez wspólnych przodków. Odwzorowujemy słownictwo genetyczne na inżynierię aktywów:

| Pojęcie INBREED | Interpretacja w energetyce |
|---|---|
| Osobnik | Konstrukcja / model transformatora |
| Rodzic (ojciec/matka) | Konstrukcja-poprzednik, od której się wywodzi |
| Generacja (CLASS) | Cykl zamówień / projektowy |
| Współczynnik inbredu *F* | Stopień samopodobnego dziedzictwa w obrębie konstrukcji |
| Współczynnik kowariancji / pokrewieństwa | Wspólne dziedzictwo konstrukcyjne między dwiema konstrukcjami |
| Najmniej spokrewniona para | Najlepsze redundantne dopasowanie dla odporności N-1 |

## Krok 1 — Zdefiniowanie rodowodu konstrukcyjnego

Wprowadzamy `DesignLineage` bezpośrednio, aby struktura pokrewieństwa była jednoznaczna. Istnieją trzy **nienakładające się generacje projektowe**:

- **Generacja 1** — cztery założycielskie platformy (`G1_P1`-`G1_P4`) bez poprzedników (puste pola rodziców).
- **Generacja 2** — cztery konstrukcje pochodne (`G2_D1`-`G2_D4`), z których każda jest zaprojektowana na bazie dwóch **odrębnych** platform Generacji 1. `G2_D1` i `G2_D2` to *pełne rodzeństwo* (obie od `G1_P1` x `G1_P2`); `G2_D3` i `G2_D4` to pełne rodzeństwo (obie od `G1_P3` x `G1_P4`).
- **Generacja 3** — cztery obecne modele flotowe (`G3_T1`-`G3_T4`). `G3_T1` jest zbudowany z pary rodzeństwa `G2_D1` x `G2_D2`, a `G3_T3` z pary rodzeństwa `G2_D3` x `G2_D4`; te dwie konstrukcje koncentrują więc dziedzictwo. `G3_T2` i `G3_T4` każdy krzyżuje dwie rozłączne linie.

Kolumny `ID`, `Parent1` i `Parent2` niosą rodowód; `Sex` zapisuje, która linia inżynierska prowadziła konstrukcję. Krótki kolejny krok DATA zamienia symbol zastępczy `.` na puste pole, tak aby platformy założycielskie miały puste pola rodziców, zgodnie z oczekiwaniami PROC INBREED.

In [1]:
DANE DesignLineage;
   DŁUGOŚĆ ID $12 Parent1 $12 Parent2 $12 Sex $1;
   INFILE DATALINES truncover;
   WEJŚCIE Generation ID $ Parent1 $ Parent2 $ Sex $;
   ETYKIETA Generation="Generacja"
      ID="Projekt"
      Parent1="Poprzednik 1"
      Parent2="Poprzednik 2"
      Sex="Linia wiodąca";
   DATALINES;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
WYKONAJ;

/* Platformy założycielskie nie mają poprzedników: zamień symbol zastępczy na puste pole */
DANE DesignLineage;
   USTAW DesignLineage;
   JEŚLI Parent1 = '.' WTEDY Parent1 = ' ';
   JEŚLI Parent2 = '.' WTEDY Parent2 = ' ';
WYKONAJ;

TYTUŁ 'Rodowód projektowy transformatora';
PROCEDURA DRUKUJ DANE=DesignLineage noobs ETYKIETA;
   ZMIENNA Generation ID Parent1 Parent2 Sex;
WYKONAJ;

                                           Rodowód projektowy transformatora                                            


Generacja  Projekt  Poprzednik 1  Poprzednik 2   Linia wiodąca
---------  -------  ------------  ------------  --------------
        1  G1_P1                                M
        1  G1_P2                                M
        1  G1_P3                                M
        1  G1_P4                                F
        2  G2_D1    G1_P1         G1_P2         M
        2  G2_D2    G1_P1         G1_P2         F
        2  G2_D3    G1_P3         G1_P4         F
        2  G2_D4    G1_P3         G1_P4         M
        3  G3_T1    G2_D1         G2_D2         M
        3  G3_T2    G2_D1         G2_D3         F
        3  G3_T3    G2_D3         G2_D4         F
        3  G3_T4    G2_D2         G2_D4         M




NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to Rodowód projektowy transformatora.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## Krok 2 — Współczynniki inbredu według generacji projektowej

Uruchamiamy PROC INBREED w **trybie wielogeneracyjnym**, podając `Generation` w instrukcji `CLASS`, co drukuje podsumowanie rodowodu według generacji. Instrukcja `VAR` dostarcza trzy kolumny rodowodu w wymaganej kolejności: osobnik, pierwszy poprzednik, drugi poprzednik.

- Opcja **IND** drukuje współczynnik inbredu *F* każdej konstrukcji — miarę tego, jak skoncentrowane jest jej własne dziedzictwo. Konstrukcja zbudowana z dwóch blisko spokrewnionych poprzedników niesie wysokie *F*.
- Opcja **MATRIX** drukuje pełną macierz addytywnego pokrewieństwa, dzięki czemu możemy odczytać dziedzictwo parami bezpośrednio.
- Opcja **AVERAGE** dołącza średni współczynnik inbredu dla całej floty — pojedyncze podsumowanie różnorodności konstrukcyjnej.

Wartości bliskie 0 oznaczają niezależne linie; *F* rośnie, gdy dwaj poprzednicy danej konstrukcji stają się bliżej spokrewnieni.

In [2]:
TYTUŁ 'Współczynniki inbredu według generacji projektowej';
PROCEDURA inbreed DANE=DesignLineage ind average MATRIX;
   ZMIENNA ID Parent1 Parent2;
   KLASA Generation;
WYKONAJ;

                                   Współczynniki inbredu według generacji projektowej                                   


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generacja/Class
--------------------------------------
Generacja 1            Members = 4
Generacja 2            Members = 4
Generacja 3            Members = 4

Inbreeding Coefficients (Generacja 1)
--------------------------------------
Projekt             F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (Generacja 2)
--------------------------------------
Projekt             F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficie


NOTE: Option TITLE changed to Współczynniki inbredu według generacji projektowej.
NOTE: PROC INBREED data=DesignLineage



## Krok 3 — Współczynniki kowariancji zapisane do dalszej oceny ryzyka

Zastąpienie widoku inbredu opcją **COVAR** raportuje te same relacje jako **współczynniki kowariancji (addytywnego pokrewieństwa)** — postać, którą zespół zarządzania aktywami mógłby wprowadzić do oceny ryzyka redundancji. Opcja **OUTCOV=** przechwytuje pełną macierz jako zbiór danych (`DesignCovar`), w którym kolumny `Col1`-`Col12` przechowują pokrewieństwo każdej konstrukcji do każdej innej (w kolejności rodowodu) — gotowe do połączenia z przypisaniami do rozdzielni.

Drukujemy zbiór wyjściowy, aby zapisana macierz była widoczna obok listingu.

In [3]:
TYTUŁ 'Współczynniki kowariancji (pokrewieństwa)';
PROCEDURA inbreed DANE=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   ZMIENNA ID Parent1 Parent2;
   KLASA Generation;
WYKONAJ;

TYTUŁ 'Macierz pokrewieństwa OUTCOV= zapisana do oceny ryzyka';
PROCEDURA DRUKUJ DANE=DesignCovar noobs ETYKIETA;
   ETYKIETA ID="Projekt";
WYKONAJ;

                                       Współczynniki kowariancji (pokrewieństwa)                                        


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generacja/Class
--------------------------------------
Generacja 1            Members = 4
Generacja 2            Members = 4
Generacja 3            Members = 4

Inbreeding Coefficients (Generacja 1)
--------------------------------------
Projekt             F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (Generacja 1)
--------------------------------------------------
Projekt                G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.0000
G1_


NOTE: Option TITLE changed to Współczynniki kowariancji (pokrewieństwa).
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to Macierz pokrewieństwa OUTCOV= zapisana do oceny ryzyka.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## Krok 4 — Najmniej spokrewnione pary dla instalacji redundantnych (N-1)

Zapisana macierz `DesignCovar` to dokładnie to, czego potrzebuje analiza redundancji. Pokrewieństwo konstrukcji *i* do konstrukcji *j* znajduje się w wierszu *i*, kolumnie `Col`*j* (kolumny są w kolejności rodowodu, więc `Col9`-`Col12` odpowiadają czterem obecnym modelom flotowym `G3_T1`-`G3_T4`).

Odczytujemy cztery wiersze obecnej floty z `DesignCovar`, tworzymy każdą nieuporządkowaną parę w obrębie obecnej floty, dołączamy jej współczynnik pokrewieństwa i sortujemy od najmniej spokrewnionych. Celem jest wybór par redundantnych, których wspólne dziedzictwo jest **najniższe** — minimalizuje to szansę, że jedna odziedziczona wada konstrukcyjna wyłączy obie połowy pozycji chronionej w trybie N-1.

In [4]:
/* Pobierz cztery wiersze obecnej floty; Col9..Col12 to pokrewieństwa
   do G3_T1..G3_T4 (kolejność kolumn rodowodu). Zbuduj każdą nieuporządkowaną parę. */
DANE Pairs;
   USTAW DesignCovar;
   GDZIE ID IN ('G3_T1','G3_T2','G3_T3','G3_T4');
   DŁUGOŚĆ DesignA $12 DesignB $12;
   TABLICA g3{4} Col9 Col10 Col11 Col12;
   TABLICA nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   ETYKIETA DesignA="Projekt A" DesignB="Projekt B" Relationship="Pokrewieństwo";
   DesignA = ID;
   POWTÓRZ j = 1 TO 4;
      DesignB = nm{j};
      JEŚLI DesignA < DesignB WTEDY POWTÓRZ;
         Relationship = g3{j};
         WYJŚCIE;
      KONIEC;
   KONIEC;
   ZACHOWAJ DesignA DesignB Relationship;
WYKONAJ;

PROCEDURA SORTUJ DANE=Pairs; WEDŁUG Relationship; WYKONAJ;

TYTUŁ 'Kandydujące pary redundantne (N-1), od najmniej spokrewnionych';
PROCEDURA DRUKUJ DANE=Pairs noobs ETYKIETA;
   ZMIENNA DesignA DesignB Relationship;
WYKONAJ;
TYTUŁ;

                             Kandydujące pary redundantne (N-1), od najmniej spokrewnionych                             


Projekt A  Projekt B   Pokrewieństwo
---------  ---------  --------------
G3_T1      G3_T3                   0
G3_T2      G3_T4                0.25
G3_T1      G3_T2               0.375
G3_T1      G3_T4               0.375
G3_T2      G3_T3               0.375
G3_T3      G3_T4               0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to Kandydujące pary redundantne (N-1), od najmniej spokrewnionych.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## Interpretacja wyników

**Współczynniki inbredu (Krok 2).** Wszystkie założycielskie platformy Generacji 1 oraz wszystkie pochodne Generacji 2 wykazują *F* = 0 — z założenia żadna nie ma dwóch spokrewnionych poprzedników. Dwa modele Generacji 3 wykazują *F* = 0,25: `G3_T1` (zbudowany z pary rodzeństwa `G2_D1` x `G2_D2`) i `G3_T3` (z pary rodzeństwa `G2_D3` x `G2_D4`). Ich poprzednicy wywodzą się z *tej samej* pary platform założycielskich, więc ich dziedzictwo jest skoncentrowane. Z punktu widzenia niezawodności są to konstrukcje najbardziej narażone na pojedynczą odziedziczoną wadę i wymagają dodatkowych, niezależnych testów kwalifikacyjnych. `G3_T2` i `G3_T4`, które krzyżują dwie rozłączne linie, mają *F* = 0.

**Macierz pokrewieństwa (Krok 3).** Wartości poza przekątną kwantyfikują parami wspólne dziedzictwo. Obie pary rodzeństwa Generacji 2 wykazują wzajemne pokrewieństwo 0,50 (`G2_D1`-`G2_D2` oraz `G2_D3`-`G2_D4`), podczas gdy pochodne z przeciwnych linii mają 0,00. Inbredowe konstrukcje obecnej floty mają samopokrewieństwo 1,25 (= 1 + *F*), widoczne na przekątnej. Zbiór `DesignCovar` udostępnia pełną macierz do połączenia z przypisaniami do rozdzielni.

**Najmniej spokrewnione pary (Krok 4).** Uszeregowanie każdej pary obecnej floty według pokrewieństwa stawia **`G3_T1` i `G3_T3` na pierwszym miejscu przy pokrewieństwie 0,00** — wywodzą się one z zupełnie rozłącznych platform przodków i nie dzielą żadnego dziedzictwa konstrukcyjnego. To najsilniejsze redundantne dopasowanie, szczególnie cenne, ponieważ `G3_T1` sam w sobie jest jedną z dwóch najbardziej inbredowych konstrukcji: sparowanie go z zupełnie niespokrewnionym partnerem zabezpiecza jego skoncentrowane dziedzictwo. Kolejną najlepszą parą jest `G3_T2` i `G3_T4` przy 0,25; pozostałe pary mają 0,375. Średni współczynnik inbredu floty wynoszący **0,0417** (wydrukowany przez opcję AVERAGE w Kroku 2) podsumowuje ogólną różnorodność konstrukcyjną. Dział zamówień powinien preferować pary o najniższym pokrewieństwie dla pozycji N-1 i z czasem poszerzać bazę przodków — inżynierski odpowiednik unikania wąskiego gardła genetycznego.

**Zastrzeżenie.** Są to ilustracyjne dane syntetyczne; badanie produkcyjne budowałoby rodowód na podstawie rzeczywistych zapisów rewizji konstrukcyjnych producenta i weryfikowałoby wskaźniki pokrewieństwa względem historycznych zdarzeń awarii wspólnego trybu przed podjęciem decyzji zamówieniowych.